In [ ]:
import os
from pathlib import Path
import torch
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import dualneuron
from dualneuron.screening.sets import RenderedImages, ImagenetImages
from dualneuron.twins.nets import load_model, model_summary
from dualneuron.synthesis.ascend import fourier_ascending, pixel_ascending
from dualneuron.synthesis.visualize import blend, plot_group
from dualneuron.synthesis.objectives import response_objective
from dualneuron.analysis.pca import extract_population_responses, fit_pca, get_pc_vector
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

token = os.getenv("HF_TOKEN")
data_dir = os.getenv("DATA_DIR")

In [ ]:
architecture = 'v4'
dataset = "imagenet"

if architecture == 'v4':
    model_name = "V4ColorTaskDriven"
    use_grayscale = False
    num_channels=3
    
elif architecture == 'v4g':
    model_name = "V4GrayTaskDriven"
    use_grayscale = True
    num_channels=1
    
elif architecture == 'v1':
    model_name = "V1GrayTaskDriven"
    use_grayscale = True
    num_channels=1
    
package_dir = Path(dualneuron.__file__).parent
mask_path = package_dir / "twins" / model_name / "mask.npy"
mask = np.load(mask_path)

if dataset == "rendered":
    dset = RenderedImages( 
        data_dir=data_dir + "rendered_data",
        # Transform options
        use_center_crop=True,
        use_resize_output=True,
        use_grayscale=False,
        use_normalize=True,
        use_mask=False,
        use_crop_to_mask=False,
        use_norm=False,
        use_clip=False,
        # Transform parameters
        mask=mask,
        num_channels=3,
        output_size=(100, 100),
        crop_size=236,
        bg_value=0.0,
        clip_min=0.0,
        clip_max=1.0,
        crop_padding_frac=0.1,
        norm=None,
    )
elif dataset == "imagenet":
    dset = ImagenetImages(
        data_dir=data_dir + "datasets",
        token=token,
        split='train',
        # Transform options
        use_center_crop=True,
        use_resize_output=True,
        use_grayscale=False,
        use_normalize=True,
        use_mask=False,
        use_crop_to_mask=False,
        use_norm=False,
        use_clip=False,
        # Transform parameters
        mask=mask,
        num_channels=3,
        output_size=(100, 100),
        crop_size=236,
        bg_value=0.0,
        clip_min=0.0,
        clip_max=1.0,
        crop_padding_frac=0.1,
        norm=None,
    )

In [ ]:
if False:
    model, hooks = model_summary(
        architecture='v4', 
        input_size=(1, 3, 100, 100), 
        device='cuda'
    )

function = load_model(
    architecture='v4', 
    layer=None, 
    ensemble=False, 
    centered=True, 
    untrained=False,
    device='cuda'
)

In [ ]:
responses, indices = extract_population_responses(
    model=function,
    dataset=dset,
    batch_size=64,
    num_images=1000,
    device='cuda',
    seed=42,
    model_kwargs={'multiplex': True}
)

In [ ]:
pca_result = fit_pca(responses, n_components=3, center=True, device='cuda')
pca_result['explained_variance_ratio']

In [ ]:
target = get_pc_vector(pca_result=pca_result, pc_index=2, as_tensor=True, device='cuda')

In [ ]:
results = []

for weight in [-1, 1]:
    obj = response_objective(
        function, 
        target=target, 
        mode='project', 
        sign=weight, 
        loss_type='mse',
        model_kwargs={'multiplex': True},
    )
    
    result = fourier_ascending(
        objective_function=obj,
        magnitude_path='natural_rgb.npy',
        image_size=100,
        init_image=None,
        total_steps=512,
        learning_rate=1.0,
        lr_schedule=True,
        eta_min=0.0,
        noise=0.0,
        values_range=(-2.0, 2.0),
        range_fn='tanh',
        nb_crops=1,
        box_size=(1.0, 1.0),
        target_norm=None,
        tv_weight=1e-5,
        jitter_std=0.0,
        oversample=1, 
        reflect_pad_frac=0.0,
        device='cuda',
        verbose=True,
        save_all_steps=True,
    )
    results.append(result)

In [ ]:
poles = []

for result in results:
    image = result['image'][-1]
    alpha = result['alpha'][-1]
    pole = blend(image, alpha, alphacut=0.0, boost=1.0, bg_value=0.5)
    poles.append(pole)

plot_group(poles, cols=2)